# Baseline model (ECG) for brain prediction 

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch

DATA = Path("../data")
OUT = Path("../outputs")
OUT.mkdir(exist_ok=True)
FS = 64 # samples/second
HEARTCHANNELS = ["ECG"]

splits = {}
for s in ("train", "val", "test"):
    seg = torch.load(DATA / f"segments_h1k_{s}.pt", weights_only=True)
    splits[s] = pd.DataFrame({"id": [t[4].removesuffix(".edf") for t in seg],
                              "age": [t[3] for t in seg]})

In [15]:
def load_ecg(rec_id):
    x = torch.load(DATA / "pre_processed_ecg" / f"{rec_id}_ecg.pt", weights_only=True)
    x = x.float().numpy() 
    last = np.flatnonzero(np.any(x != 0, axis=0))[-1]   # last sample where any channel is non-zero
    return x[:, : last + 1]                             # drops the zero-padded tail

def make_windows(x, win_s, start_s, end_s):
    seg = x[:, start_s * FS : end_s * FS]
    n = win_s * FS                                      # samples per window
    k = seg.shape[1] // n                               # whole windows that fit
    return seg[:, : k * n].reshape(x.shape[0], k, n).transpose(1, 0, 2)   # (windows, channels, samples)

In [16]:
from scipy.signal import welch

BANDS = {"delta": (0.5, 4), "theta": (4, 8), "alpha": (8, 12), "sigma": (12, 16), "beta": (16, 30)}
EPS = 1e-6   # avoids log(0) and division by zero

def window_features(w):
    # Hjorth parameters, from the signal and its first and second differences
    d1 = np.diff(w, axis=-1)
    d2 = np.diff(d1, axis=-1)
    var, var1, var2 = w.var(-1), d1.var(-1), d2.var(-1)
    mobility = np.sqrt(var1 / (var + EPS))
    complexity = np.sqrt(var2 / (var1 + EPS)) / (mobility + EPS)
    feats = {"logvar": np.log(var + EPS), "mobility": mobility, "complexity": complexity}

    # Band powers: Welch spectrum from 4 s segments (0.25 Hz resolution)
    freqs, psd = welch(w, fs=FS, nperseg=4 * FS, axis=-1)
    total = psd[..., (freqs >= 0.5) & (freqs < 30)].sum(-1)
    for name, (lo, hi) in BANDS.items():
        power = psd[..., (freqs >= lo) & (freqs < hi)].sum(-1)
        feats[f"log{name}"] = np.log(power + EPS)
        feats[f"rel{name}"] = power / (total + EPS)
    return feats   # 13 arrays, each of shape (windows, channels)

In [ ]:
from tqdm import tqdm

PARTS = {"preN2": (0, 300), "N2": (300, 900)}

def recording_features(rec_id, win_s=30):
    x = load_ecg(rec_id)
    row = {"id": rec_id}
    for part, (start, end) in PARTS.items():
        feats = window_features(make_windows(x, win_s, start, end))
        for name, values in feats.items():
            medians = np.median(values, axis=0)          # median over windows, one value per channel
            for ch, value in zip(HEARTCHANNELS, medians):
                row[f"{part}_{name}_{ch}"] = value
    return row

all_ids = pd.concat(splits.values()).id
features = pd.DataFrame([recording_features(r) for r in tqdm(all_ids)]).set_index("id")
features.to_csv(OUT / "features_win30_ECG.csv")
# later: features = pd.read_csv(OUT / "features_win30.csv", index_col="id")

100%|██████████| 1032/1032 [00:18<00:00, 56.28it/s]


In [20]:
from sklearn.metrics import r2_score

EDGES = [0, 1, 3, 6, 12, np.inf]
GROUPS = ["<1", "1-3", "3-6", "6-12", "12+"]

def evaluate(age, pred):
    err = np.abs(pred - age)
    group = pd.cut(age, EDGES, right=False, labels=GROUPS)
    per_group = pd.Series(err).groupby(group, observed=False).mean()
    return {"MAE": err.mean(), "balanced MAE": per_group.mean(), **per_group.to_dict(),
            "r": np.corrcoef(age, pred)[0, 1], "R2": r2_score(age, pred)}

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV

def xy(split, columns):
    df = splits[split]
    return features.loc[df.id, columns].to_numpy(), df.age.to_numpy()

results = {}

def run(name, columns, model, log_target=True):
    X_tr, y_tr = xy("train", columns)
    X_va, y_va = xy("val", columns)
    model.fit(X_tr, np.log(y_tr) if log_target else y_tr)
    pred = model.predict(X_va)
    if log_target:
        pred = np.exp(pred) # back to years before scoring
    results[name] = evaluate(y_va, pred)
    return pred

def ridge():
    # RidgeCV picks the regularisation strength by leave-one-out on train, so val stays untouched
    return make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 4, 25)))

ALL = list(features.columns)

y_tr, y_va = splits["train"].age.to_numpy(), splits["val"].age.to_numpy()
results["predict median"] = evaluate(y_va, np.full(len(y_va), np.median(y_tr)))
pred = run("ridge, ECG channel", ALL, ridge())
(OUT / "ridge_ecg").mkdir(parents=True, exist_ok=True)
pd.DataFrame({"id": splits["val"].id, "age": y_va, "pred": pred}).to_csv(OUT / "ridge_ecg" / "val_predictions.csv", index=False)
run("ridge, ECG pre-N2 only", [c for c in ALL if c.startswith("preN2_")], ridge())
run("ridge, ECG N2 only", [c for c in ALL if c.startswith("N2_")], ridge())

pd.DataFrame(results).T.round(2)

,MAE,balanced MAE,<1,1-3,3-6,6-12,12+,r,R2
predict median,2.98,4.37,2.47,1.18,1.63,5.51,11.07,0.00,-0.04
"ridge, all channels",2.08,3.03,0.96,1.12,2.35,3.54,7.19,0.62,0.37
"ridge, ECG pre-N2 only",2.15,3.21,1.01,1.06,2.57,3.46,7.94,0.57,0.30
"ridge, ECG N2 only",2.13,3.05,1.15,1.06,2.40,3.63,7.01,0.61,0.34
